# DengueRadar - ETL e Limpeza de Dados
Este notebook é responsável por carregar os dados brutos do DATASUS e do INMET, tratar as datas, organizar as semanas epidemiológicas e cruzar as informações para treinar a IA.

In [6]:
import pandas as pd
import numpy as np
import os

# Criando as pastas automaticamente caso você não tenha criado no VS Code
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
print('Pastas prontas!')

Pastas prontas!


## 1. Tratamento dos Dados da Dengue (DATASUS)
Vamos usar o `.melt()` do Pandas para transformar essas colunas em linhas e converter tudo para números!

In [7]:
# Carregando o arquivo do DATASUS
# Usamos skipfooter=10 para pular as notas de rodapé que o TabNet coloca no final
df_dengue = pd.read_csv('../data/raw/datasus_dengue.csv', sep=';', encoding='latin1', skipfooter=10, engine='python')

# Pegando as colunas que representam os anos (ignorando a primeira coluna e o 'Total')
colunas_anos = [col for col in df_dengue.columns if col.isnumeric()]
coluna_semana = df_dengue.columns[0]

# Transformando de 'Largo' (wide) para 'Longo' (long)
df_dengue_long = pd.melt(df_dengue, id_vars=[coluna_semana], value_vars=colunas_anos, var_name='ano', value_name='casos')

# Extraindo apenas o número da semana (ex: 'Semana 01' -> 1)
df_dengue_long['semana_epidem'] = df_dengue_long[coluna_semana].str.extract(r'(\d+)').astype(float)

# Substituindo o '-' por 0 e convertendo para número
df_dengue_long['casos'] = pd.to_numeric(df_dengue_long['casos'].replace('-', 0)).fillna(0)
df_dengue_long['ano'] = df_dengue_long['ano'].astype(int)

# Removendo linhas inválidas
df_dengue_long = df_dengue_long.dropna(subset=['semana_epidem'])
df_dengue_long['semana_epidem'] = df_dengue_long['semana_epidem'].astype(int)

# Removendo a coluna de texto original da semana
df_dengue_long = df_dengue_long.drop(columns=[coluna_semana])

print("Dados do DATASUS Limpos:")
display(df_dengue_long.head())

Dados do DATASUS Limpos:


,ano,casos,semana_epidem
0,2024,179,1
1,2024,0,2
2,2024,0,3
3,2024,0,4
4,2024,0,5


## 2. Tratamento dos Dados de Clima (INMET)
Os dados do INMET são diários e variam por estação. A ideia aqui é carregar, agrupar pela semana epidemiológica e tirar a média térmica e soma da chuva.

In [11]:
print(df_clima.columns.tolist())

['Data', 'Hora UTC', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)', 'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)', 'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)', 'RADIACAO GLOBAL (Kj/m²)', 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)', 'TEMPERATURA DO PONTO DE ORVALHO (°C)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)', 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)', 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)', 'VENTO, VELOCIDADE HORARIA (m/s)', 'Unnamed: 19', 'ano', 'semana_epidem']


In [14]:
# 1. Carregar os dados (com indicação do decimal)
df_clima = pd.read_csv('../data/raw/inmet_clima.csv', sep=';', encoding='latin1', skiprows=8, decimal=',')

# 2. Converter a data
df_clima['Data'] = pd.to_datetime(df_clima['Data'])
df_clima['ano'] = df_clima['Data'].dt.year
df_clima['semana_epidem'] = df_clima['Data'].dt.isocalendar().week

# 3. Forçar as colunas a serem reconhecidas como números
col_temp = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
col_chuva = 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'

# Substitui possíveis vírgulas residuais e ignora os erros textuais das medições do sensor
df_clima[col_temp] = pd.to_numeric(df_clima[col_temp].astype(str).str.replace(',', '.'), errors='coerce')
df_clima[col_chuva] = pd.to_numeric(df_clima[col_chuva].astype(str).str.replace(',', '.'), errors='coerce')

# 4. Agrupar tirando a média e a soma
clima_semanal = df_clima.groupby(['ano', 'semana_epidem']).agg({
    col_temp: 'mean',
    col_chuva: 'sum'
}).reset_index()

# 5. Renomear as colunas
clima_semanal = clima_semanal.rename(columns={
    col_temp: 'temp_media',
    col_chuva: 'precipitacao_total'
})

print("Dados do Clima Agrupados e Renomeados:")
display(clima_semanal.head())

Dados do Clima Agrupados e Renomeados:


,ano,semana_epidem,temp_media,precipitacao_total
0,2025,1,24.644792,57.8
1,2025,2,21.808929,12.4
2,2025,3,23.032738,40.6
3,2025,4,25.540719,180.6
4,2025,5,22.278571,150.8


## 3. O Cruzamento e Exportação
Juntando Saúde + Clima em uma única tabela para a IA.

In [15]:
df_final = pd.merge(clima_semanal, df_dengue_long, on=['ano', 'semana_epidem'], how='inner')

df_final.to_csv('../data/processed/dataset_dengueradar_limpo.csv', index=False)
print("Sucesso! Base do DengueRadar unificada e pronta para o Scikit-Learn!")

Sucesso! Base do DengueRadar unificada e pronta para o Scikit-Learn!
